# Notebook 5 — Text-to-Change Search with ChangeCLIP-style embeddings

### *Hands-On Session: Earth Embeddings for EO — Retrieval, Discovery, and Change-Oriented Search*

**Instructors:** Fuxun Yu, Rishi Madhok (TerraByte AI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iceysteel/ESA-NASA-Workshop-2026/blob/main/Day%202/Track%202/Earth-Embeddings-EO/notebook_5_text_to_change_search.ipynb)

---

## What we're doing

Notebooks 3 and 4 retrieved *single scenes* from text or image queries. Now we move to the **change-oriented** setting: the pool isn't single images, it's **bi-temporal pairs** (before / after). The query is still English. The system must return *the pairs where the described change actually happened*.

This is the day-to-day pattern for:
- *"Find me locations where new buildings were constructed between these two acquisitions."*
- *"Surface pairs where vegetation was cleared."*
- *"Where did infrastructure appear in this scene?"*

## The model: ChangeCLIP-style retrieval with Git-RSCLIP

[ChangeCLIP (Dong et al., ISPRS 2024)](https://github.com/dyzy41/ChangeCLIP) takes a frozen CLIP vision tower, encodes each frame of a bi-temporal pair, fuses them with a learned decoder, and predicts a change segmentation mask. The paper's key insight is that **CLIP image features are already in the same embedding space as CLIP text features** — so the visual side starts text-aligned, and you don't need a separately trained text encoder for change retrieval.

We adopt that insight but swap the backbone for [`lcybuaa/Git-RSCLIP-base`](https://huggingface.co/lcybuaa/Git-RSCLIP-base) — a SigLIP-style CLIP pretrained on 10 M remote-sensing image-text pairs. ChangeCLIP uses OpenAI CLIP, but OpenAI CLIP barely saw satellite data; on small EO change events Git-RSCLIP's RS-aware visual features give a much stronger retrieval signal. Same model NB3 and NB4 used.

**Pipeline:**

1. Encode each bi-temporal pair as **two Git-RSCLIP image vectors** (t1 and t2).
2. Define `pair_embedding = normalize(t2 − t1)` — a *change vector* in shared image/text space.
3. Encode the user's English query with the **Git-RSCLIP text encoder**.
4. Rank pairs by cosine similarity between the change vector and the text vector.

Skipping ChangeCLIP's bi-temporal *decoder* costs us per-pixel change masks but keeps the retrieval lightweight, fully text-driven, and runnable on Colab CPU.

## The data: LEVIR-CD (Google Earth, 0.5 m, building change)

[LEVIR-CD (Chen & Shi 2020)](https://justchenhao.github.io/LEVIR/) ships 637 bi-temporal 1024×1024 Google Earth pairs at **0.5 m / pixel**, each annotated with a binary building-change mask. We use 445 pairs from the train split — large enough for retrieval signal, small enough to encode once in under a minute. At 0.5 m, individual buildings cover hundreds of pixels: Git-RSCLIP can actually *see* them, which makes the `t2 − t1` difference vector carry real semantic load (a sister Sentinel-2 dataset at 10 m, where buildings are sub-pixel, retrieval works much worse — exactly the kind of failure mode you want to internalise).


## Runtime

**CPU is fine** — the heavy lifting (encoding 500 pairs × 2 frames with CLIP) ran offline. At workshop time the notebook only encodes the **text query** (~200 ms on CPU) and runs a single 500×512 mat-vec.


In [ ]:
!pip install -q transformers huggingface_hub datasets


In [ ]:
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModel, AutoProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


## 1. Download the precomputed bi-temporal embedding pool

445 bi-temporal pairs from LEVIR-CD train, each encoded as two 768-d Git-RSCLIP vectors (L2-normalized). Hosted on the Hub at [`zterrabyte/levircd-changeclip-embeddings`](https://huggingface.co/datasets/zterrabyte/levircd-changeclip-embeddings); `mask_pct` is the fraction of the change mask labelled as *changed* (useful for sanity-checking retrieval).


In [ ]:
HF_POOL_REPO = 'zterrabyte/levircd-changeclip-embeddings'
HF_POOL_FILE = 'levircd_changeclip_pairs.npz'

t0 = time.time()
npz_path = hf_hub_download(repo_id=HF_POOL_REPO, filename=HF_POOL_FILE, repo_type='dataset')
print(f'Downloaded in {time.time()-t0:.1f}s — {Path(npz_path).stat().st_size/1e6:.1f} MB')

blob = np.load(npz_path, allow_pickle=True)
pair_emb = blob['embeddings'].astype(np.float32)        # (N, 2, 512)
image_names = [str(n) for n in blob['image_names']]
mask_pct = blob['mask_pct']
sampled_indices = blob['sampled_indices']
N = pair_emb.shape[0]
print(f'Pool: {pair_emb.shape}  (N={N} pairs, t1/t2 × 768 dim)')
print(f'mask_pct: median={np.median(mask_pct):.3%}  max={mask_pct.max():.2%}')


## 2. Build the *change vector* for every pair

The whole notebook hinges on this one line:

```python
change_vec = normalize(t2 − t1)
```

Why this is sensible: CLIP image features are unit-norm vectors on a hypersphere. Two acquisitions of the *same* scene with no change → near-identical vectors → tiny difference, near zero magnitude. A scene where something appeared / disappeared → the difference vector points toward whatever changed, *in CLIP space*. Because CLIP's text encoder shares that space, English descriptions of the change project onto the same axes.


In [ ]:
t1 = pair_emb[:, 0, :]                                 # (N, 512)
t2 = pair_emb[:, 1, :]                                 # (N, 512)
raw_diff = t2 - t1                                      # (N, 512)  — points 'toward the change'
diff_norm = np.linalg.norm(raw_diff, axis=1)
change_vec = raw_diff / (diff_norm[:, None] + 1e-12)   # L2-normalized

print(f'||t2 - t1|| stats:  min={diff_norm.min():.3f}  median={np.median(diff_norm):.3f}  max={diff_norm.max():.3f}')
# Sanity: pairs with bigger ground-truth change masks should tend to have larger diff norms.
from scipy.stats import spearmanr
rho, p = spearmanr(diff_norm, mask_pct)
print(f'Spearman(||diff||, mask_pct) = {rho:+.3f}  (p={p:.1e})  — positive = larger CLIP-diff for bigger labelled change')


## 3. Load the Git-RSCLIP text tower

Same checkpoint that produced the precomputed image side — guarantees shared embedding space. Same model NB3 / NB4 used.

*First run downloads ~870 MB of weights; 20–60 s on Colab.*


In [ ]:
!pip install -q sentencepiece


In [ ]:
MODEL_ID = 'lcybuaa/Git-RSCLIP-base'
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).eval().to(device)
print(f'Git-RSCLIP loaded in {time.time()-t0:.1f}s '
      f'({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)')


## 4. Pull the raw LEVIR-CD pair images (for display)

We need the actual before/after tiles to *show* the retrieved pairs. We pull the same indices from the LEVIR-CD train split so the row order matches our embeddings.


In [ ]:
t0 = time.time()
ds = load_dataset('EVER-Z/torchange_levircd', split='train')
pool_rows = [ds[int(i)] for i in sampled_indices]
print(f'Loaded {len(pool_rows)} pairs (t1+t2+mask) in {time.time()-t0:.1f}s')
assert len(pool_rows) == N, 'pool length mismatch'


## 5. The text-to-change search function

Mechanically identical to NB3 — but instead of ranking single-image embeddings we rank *difference* embeddings.


In [ ]:
def encode_text(query: str) -> np.ndarray:
    inputs = processor(text=[query], padding='max_length', return_tensors='pt').to(device)
    with torch.inference_mode():
        t = model.text_model(**inputs).pooler_output
    return F.normalize(t.float(), dim=-1).cpu().numpy()[0]

def search_change(query: str, k: int = 6, mask_pct_floor: float = 0.0):
    t0 = time.time()
    qvec = encode_text(query)
    enc_ms = (time.time() - t0) * 1000
    sims = change_vec @ qvec
    # Optional filter: drop pairs labelled as essentially no-change, so we don't show noise.
    mask = mask_pct >= mask_pct_floor
    order = np.argsort(-np.where(mask, sims, -np.inf))[:k]
    return order, sims[order], enc_ms

def show_results(query: str, k: int = 6, mask_pct_floor: float = 0.0):
    top, scores, ms = search_change(query, k, mask_pct_floor)
    fig, axes = plt.subplots(k, 3, figsize=(7.5, 2.4 * k))
    if k == 1: axes = axes[None, :]
    for row, (idx, score) in enumerate(zip(top, scores)):
        r = pool_rows[idx]
        axes[row, 0].imshow(r['t1_image']);    axes[row, 0].set_title('t1 (before)', fontsize=9); axes[row, 0].axis('off')
        axes[row, 1].imshow(r['t2_image']);    axes[row, 1].set_title('t2 (after)',  fontsize=9); axes[row, 1].axis('off')
        axes[row, 2].imshow(r['change_mask'], cmap='gray');
        axes[row, 2].set_title(f'gt change mask\n({mask_pct[idx]:.1%} changed)', fontsize=9); axes[row, 2].axis('off')
        axes[row, 0].set_ylabel(f'cos={score:.3f}', fontsize=10, color='C3')
    plt.suptitle(f'Query: "{query}"   (text encode: {ms:.0f} ms)', y=1.005, fontsize=11)
    plt.tight_layout(); plt.show()


## 6. Demo queries

LEVIR-CD is a *building-change* dataset (new construction, demolition, urban expansion), so prompts that describe those events should work; abstract or off-domain prompts let you see how the model degrades.


**Lead query — the cleanest signal we get on this pool:**


In [ ]:
show_results('a residential neighborhood was built', k=4, mask_pct_floor=0.01)


**Variant phrasings** — same underlying concept, slightly different cosines and rankings:


In [ ]:
show_results('construction of new houses', k=4, mask_pct_floor=0.01)


In [ ]:
show_results('new buildings have been constructed', k=4, mask_pct_floor=0.01)


### Opposite-direction query (vector arithmetic in action)

Because the change embedding is `normalize(t2 − t1)`, a query that reverses the temporal narrative should pull pairs from the **opposite** end of the change axis. In LEVIR-CD the labelled change is overwhelmingly *construction* (new buildings appearing between t1 and t2). So a *demolition* query should pull the rare reverse-direction pairs — and a *no-change* query should land near the origin (low absolute cosine either way).


In [ ]:
show_results('buildings were demolished', k=3, mask_pct_floor=0.0)


In [ ]:
show_results('nothing changed in the scene', k=3, mask_pct_floor=0.0)


### Try your own change query

Edit the string below and re-run. Things worth trying:

- **Direction matters.** `'a building was demolished'` vs `'a building was constructed'` should pull *opposite* ends of the change axis (one is roughly `+(t2-t1)`, the other `-(t2-t1)`).
- **Compositional queries.** `'rural land became suburban'`, `'a road appeared cutting through farmland'`.
- **Calibration probes.** `'nothing changed'` should *not* score high on pairs with large `mask_pct`.


In [ ]:
show_results('your change description here', k=4, mask_pct_floor=0.0)


## 7. Quantitative sanity — does retrieval correlate with the ground-truth change masks?

LEVIR-CD gives us a binary change mask per pair. There is no per-pair *text label*, so we can't compute Precision@K on semantics directly. But we can ask a weaker question: **for a generic 'something was built' query, do retrieved pairs have larger labelled change masks than the unranked pool?**


In [ ]:
query = 'a residential neighborhood was built'
qvec = encode_text(query)
sims = change_vec @ qvec
order = np.argsort(-sims)
print(f'Query: {query!r}')
for K in [10, 25, 50, 100]:
    top_mask = mask_pct[order[:K]].mean()
    bot_mask = mask_pct[order[-K:]].mean()
    print(f'  top-{K:>3}  mean mask_pct = {top_mask:.2%}     '
          f'bot-{K:>3}  mean mask_pct = {bot_mask:.2%}     '
          f'ratio = {top_mask / (bot_mask + 1e-9):>5.2f}x')
print(f'\noverall mean mask_pct = {mask_pct.mean():.2%}')


## 8. Takeaways

- **Change retrieval = arithmetic on CLIP-aligned features.** No bespoke change-detection model is needed for retrieval — a frozen CLIP image tower already lands every frame on the text manifold. `normalize(t2 − t1)` is a directional change vector you can index, rank, and combine arithmetically.
- **What ChangeCLIP adds on top** is a *decoder* that turns those fused features into pixel-accurate change masks. Useful when you need a localized output; not strictly required for *finding which pairs to look at*.
- **Sign matters.** `'something appeared'` and `'something disappeared'` pull opposite ends of the same axis. This is also the building block for `Q = +text_A − text_B` composition (e.g. *retrieve change pairs that look like A but not B*).
- **Domain still matters.** LEVIR-CD is building change in urban Texas / Beijing. Queries about wildfires or floods will retrieve *the closest thing in this pool*, not actual wildfires — the same open-vocabulary blur as NB3.
- **Resolution dominates this kind of retrieval.** A sister Sentinel-2 dataset (S2Looking, 10 m) returns *much* weaker retrievals on the same pipeline — buildings end up sub-pixel and the difference vector mostly captures atmospheric drift. The lesson: a CLIP-diff retriever is only as good as the spatial scale at which the change is visible in the embedding-time crops (here, Git-RSCLIP feeds 224×224 to the vision tower).
- **Where this is heading.** Notebooks 3, 4, and 5 covered single-image text retrieval, single-image visual retrieval, and bi-temporal text retrieval — all with the same `embeddings @ query_vec` core. In production this scales via FAISS / ScaNN and combines positive + negative queries for `+constructed −demolished` operations.
